<a href="https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My baseline is a transparent, manually weighted review rule. It prioritizes pages that had substantial demand before the decision moment, were in an actionable average-position range, had relatively weak CTR, or had unstable positions.

The rule adds fixed points:

- **4 points — `high_demand`:** historical impressions per day are in the top 25% of the eligible March slice.
- **3 points — `position_opportunity`:** historical average position is between 4 and 20.
- **2 points — `low_ctr`:** historical CTR is in the bottom 25% of the eligible March slice.
- **1 point — `position_instability`:** historical position variation is in the top 25%.

The total is converted to a 0–100 baseline score. Ties are resolved using historical impressions per day and clicks per day.

The score uses only information measured from 2026-03-01 through 2026-03-21. The future outcome from 2026-03-22 through 2026-03-31 is used only to evaluate Precision@20 and is never used to calculate the score.

This baseline ranks pages for human review. It does not claim that refreshing a selected page would cause recovery.

In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy

import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

# ---------------------------------------------------------
# 1. Secure Hugging Face connection
# ---------------------------------------------------------

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

assert HF_TOKEN, (
    "HF_TOKEN is missing. Add it in Colab Secrets and "
    "enable notebook access."
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

# ---------------------------------------------------------
# 2. Rebuild the same leakage-safe frame used in ML-04
# ---------------------------------------------------------

feature_sql = f"""
WITH aggregated AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(
            DISTINCT CASE
                WHEN report_date < DATE '2026-03-22'
                THEN report_date
            END
        ) AS hist_days,

        COUNT(
            DISTINCT CASE
                WHEN report_date >= DATE '2026-03-22'
                THEN report_date
            END
        ) AS outcome_days,

        SUM(
            CASE
                WHEN report_date < DATE '2026-03-22'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS hist_impressions,

        SUM(
            CASE
                WHEN report_date < DATE '2026-03-22'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS hist_clicks,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS label_outcome_impressions,

        SUM(
            CASE
                WHEN report_date < DATE '2026-03-22'
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN report_date < DATE '2026-03-22'
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS hist_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN report_date < DATE '2026-03-22'
                     AND gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS hist_position_sd

    FROM {MARCH_DAILY}

    WHERE ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

framed AS (
    SELECT
        *,

        1.0 * hist_impressions
            / NULLIF(hist_days, 0)
            AS hist_imp_per_day,

        1.0 * hist_clicks
            / NULLIF(hist_days, 0)
            AS hist_click_per_day,

        1.0 * hist_clicks
            / NULLIF(hist_impressions, 0)
            AS hist_ctr,

        1.0 * label_outcome_impressions
            / NULLIF(outcome_days, 0)
            AS label_outcome_imp_per_day

    FROM aggregated

    WHERE hist_days >= 14
      AND outcome_days >= 7
      AND hist_impressions >= 50
)

SELECT
    client_hash_id,
    content_hash_id,
    hist_days,
    outcome_days,

    hist_imp_per_day,
    hist_click_per_day,
    hist_ctr,
    hist_avg_position,
    hist_position_sd,

    label_outcome_imp_per_day,

    CAST(
        label_outcome_imp_per_day
        < 0.8 * hist_imp_per_day
        AS INTEGER
    ) AS is_declining_proxy

FROM framed
"""

feature_frame = con.sql(feature_sql).df()

honest_features = [
    "hist_imp_per_day",
    "hist_click_per_day",
    "hist_ctr",
    "hist_avg_position",
    "hist_position_sd",
]

baseline_frame = (
    feature_frame
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=honest_features + ["is_declining_proxy"]
    )
    .copy()
)

assert not baseline_frame.empty, "The baseline frame is empty."
assert baseline_frame["is_declining_proxy"].nunique() == 2, (
    "The evaluation label must contain both classes."
)
assert (
    baseline_frame.duplicated(
        ["client_hash_id", "content_hash_id"]
    ).sum()
    == 0
), "Duplicate client-content rows were found."

# ---------------------------------------------------------
# 3. Thresholds use only the historical feature window
# ---------------------------------------------------------

HIGH_DEMAND_THRESHOLD = float(
    baseline_frame["hist_imp_per_day"].quantile(0.75)
)

LOW_CTR_THRESHOLD = float(
    baseline_frame["hist_ctr"].quantile(0.25)
)

HIGH_VOLATILITY_THRESHOLD = float(
    baseline_frame["hist_position_sd"].quantile(0.75)
)

rule_thresholds = pd.DataFrame(
    [
        {
            "reason_code": "high_demand",
            "rule": "hist_imp_per_day >= March 75th percentile",
            "threshold": HIGH_DEMAND_THRESHOLD,
            "points": 4,
        },
        {
            "reason_code": "position_opportunity",
            "rule": "4 <= hist_avg_position <= 20",
            "threshold": "4 to 20",
            "points": 3,
        },
        {
            "reason_code": "low_ctr",
            "rule": "hist_ctr <= March 25th percentile",
            "threshold": LOW_CTR_THRESHOLD,
            "points": 2,
        },
        {
            "reason_code": "position_instability",
            "rule": "hist_position_sd >= March 75th percentile",
            "threshold": HIGH_VOLATILITY_THRESHOLD,
            "points": 1,
        },
    ]
)

print("Eligible baseline rows:", len(baseline_frame))
print("Historical feature count:", len(honest_features))
print("Duplicate client-content rows: 0")

display(rule_thresholds)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible baseline rows: 2348
Historical feature count: 5
Duplicate client-content rows: 0


,reason_code,rule,threshold,points
0,high_demand,hist_imp_per_day >= March 75th percentile,587.18985,4
1,position_opportunity,4 <= hist_avg_position <= 20,4 to 20,3
2,low_ctr,hist_ctr <= March 25th percentile,0.003042,2
3,position_instability,hist_position_sd >= March 75th percentile,4.837341,1


## 2. Build the ranked queue (writes the CSV)

I calculate the four rule conditions independently and add their fixed points. The maximum score is 100.

The score is not trained against the future label. The outcome label is attached only after ranking so that the baseline can be evaluated using Precision@20.

The exported action queue contains pseudonymized identifiers, historical signals, reason codes, confidence, and a suggested review action. It deliberately excludes the future label and all outcome-window measurements.

In [ ]:
# ---------------------------------------------------------
# 1. Calculate transparent rule flags
# ---------------------------------------------------------

ranked_queue = baseline_frame.copy()

ranked_queue["flag_high_demand"] = (
    ranked_queue["hist_imp_per_day"]
    >= HIGH_DEMAND_THRESHOLD
).astype(int)

ranked_queue["flag_position_opportunity"] = (
    ranked_queue["hist_avg_position"]
    .between(4.0, 20.0, inclusive="both")
).astype(int)

ranked_queue["flag_low_ctr"] = (
    ranked_queue["hist_ctr"]
    <= LOW_CTR_THRESHOLD
).astype(int)

ranked_queue["flag_position_instability"] = (
    ranked_queue["hist_position_sd"]
    >= HIGH_VOLATILITY_THRESHOLD
).astype(int)

# Fixed, human-written weights.
ranked_queue["baseline_rule_points"] = (
    4 * ranked_queue["flag_high_demand"]
    + 3 * ranked_queue["flag_position_opportunity"]
    + 2 * ranked_queue["flag_low_ctr"]
    + 1 * ranked_queue["flag_position_instability"]
)

ranked_queue["baseline_score"] = (
    10.0 * ranked_queue["baseline_rule_points"]
)

# ---------------------------------------------------------
# 2. Reason codes, actions, and confidence
# ---------------------------------------------------------

def build_reason_codes(row):
    reasons = []

    if row["flag_high_demand"] == 1:
        reasons.append("high_demand")

    if row["flag_position_opportunity"] == 1:
        reasons.append("position_opportunity")

    if row["flag_low_ctr"] == 1:
        reasons.append("low_ctr")

    if row["flag_position_instability"] == 1:
        reasons.append("position_instability")

    return " | ".join(reasons) if reasons else "monitor_only"


def choose_action(row):
    if (
        row["flag_high_demand"] == 1
        and row["flag_position_opportunity"] == 1
        and row["flag_low_ctr"] == 1
    ):
        return "REVIEW_SNIPPET_AND_INTENT"

    if (
        row["flag_high_demand"] == 1
        and row["flag_position_instability"] == 1
    ):
        return "REVIEW_REFRESH_OR_PROTECT"

    if (
        row["flag_high_demand"] == 1
        and row["flag_position_opportunity"] == 1
    ):
        return "REVIEW_CONTENT_EXPANSION"

    if row["flag_high_demand"] == 1:
        return "MONITOR_HIGH_DEMAND"

    return "MONITOR"


def choose_confidence(row):
    if row["baseline_rule_points"] >= 8:
        return "high"

    if row["baseline_rule_points"] >= 5:
        return "medium"

    return "low"


ranked_queue["reason_codes"] = ranked_queue.apply(
    build_reason_codes,
    axis=1,
)

ranked_queue["suggested_action"] = ranked_queue.apply(
    choose_action,
    axis=1,
)

ranked_queue["confidence"] = ranked_queue.apply(
    choose_confidence,
    axis=1,
)

# Deterministic ranking and tie-breaking.
ranked_queue = (
    ranked_queue
    .sort_values(
        by=[
            "baseline_score",
            "hist_imp_per_day",
            "hist_click_per_day",
            "content_hash_id",
        ],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "rank",
    np.arange(1, len(ranked_queue) + 1),
)

# ---------------------------------------------------------
# 3. Evaluate Precision@20 against the future proxy
# ---------------------------------------------------------

K = min(20, len(ranked_queue))

base_rate = float(
    ranked_queue["is_declining_proxy"].mean()
)

precision_at_20 = float(
    ranked_queue
    .head(K)["is_declining_proxy"]
    .mean()
)

lift_at_20 = (
    precision_at_20 / base_rate
    if base_rate > 0
    else np.nan
)

baseline_metrics = pd.DataFrame(
    [
        {
            "metric": "eligible_rows",
            "value": len(ranked_queue),
        },
        {
            "metric": "positive_base_rate",
            "value": base_rate,
        },
        {
            "metric": "precision_at_20",
            "value": precision_at_20,
        },
        {
            "metric": "lift_at_20_vs_base_rate",
            "value": lift_at_20,
        },
    ]
)

display(baseline_metrics.round(4))

# ---------------------------------------------------------
# 4. Write the operational action queue
# ---------------------------------------------------------

# Do not export the future label or outcome-window columns.
queue_export_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "baseline_rule_points",
    "suggested_action",
    "reason_codes",
    "confidence",
    "hist_imp_per_day",
    "hist_click_per_day",
    "hist_ctr",
    "hist_avg_position",
    "hist_position_sd",
]

queue_export = ranked_queue[
    queue_export_columns
].copy()

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    OUTPUT_DIR / "baseline_action_score.csv"
)

queue_export.to_csv(
    OUTPUT_PATH,
    index=False,
)

assert OUTPUT_PATH.exists()
assert "is_declining_proxy" not in queue_export.columns
assert "label_outcome_imp_per_day" not in queue_export.columns

print("Queue rows written:", len(queue_export))
print("CSV path:", OUTPUT_PATH)
print("CSV size in bytes:", OUTPUT_PATH.stat().st_size)

display(queue_export.head(10))

,metric,value
0,eligible_rows,2348.0000
1,positive_base_rate,0.2355
2,precision_at_20,0.3500
3,lift_at_20_vs_base_rate,1.4861


Queue rows written: 2348
CSV path: work/outputs/baseline_action_score.csv
CSV size in bytes: 440969


,rank,client_hash_id,content_hash_id,baseline_score,baseline_rule_points,suggested_action,reason_codes,confidence,hist_imp_per_day,hist_click_per_day,hist_ctr,hist_avg_position,hist_position_sd
0,1,client_20259bd6705d81d4,content_7fa5790abace4589,100.0,10,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high,1185.058824,1.882353,0.001588,4.632731,6.141777
1,2,client_23a62021009f63c4,content_e7f8bc973c5204f7,100.0,10,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high,1087.000000,2.200000,0.002024,19.452714,5.809662
2,3,client_23a62021009f63c4,content_bec346091ea3e9e7,100.0,10,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high,1057.900000,0.950000,0.000898,19.051092,7.750095
3,4,client_23a62021009f63c4,content_496ecabe1eae48bb,100.0,10,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high,938.700000,2.500000,0.002663,11.441781,5.359805
4,5,client_23a62021009f63c4,content_7c142ec4c5fddafb,100.0,10,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high,781.952381,1.809524,0.002314,17.746422,5.392690
5,6,client_23a62021009f63c4,content_4a48f1c5632e340b,100.0,10,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high,591.428571,1.000000,0.001691,5.893357,9.607991
6,7,client_23a62021009f63c4,content_e8a52cf3d5988c07,90.0,9,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr,high,8525.000000,22.476190,0.002637,15.593699,1.640181
7,8,client_23a62021009f63c4,content_5e1c049f62e33b11,90.0,9,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr,high,4352.850000,6.650000,0.001528,17.858690,1.519467
8,9,client_e547b89c05043229,content_8e5fefdae6cea24b,90.0,9,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr,high,2711.619048,6.380952,0.002353,4.341792,1.303292
9,10,client_e547b89c05043229,content_77276ad7a26f4905,90.0,9,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr,high,2656.150000,5.950000,0.002240,4.060332,0.902281


## 3. Top-20 review

The table below reviews the first 20 ranked pages individually.

For every page it shows:

- the suggested human action;
- the reason codes that produced the score;
- a confidence note;
- the observed future-decline proxy used only for retrospective evaluation;
- what could make the recommendation wrong.

A proxy-positive result means the page declined under my defined short future window. It does not prove that refreshing the page would cause recovery.

In [ ]:
# ---------------------------------------------------------
# Top-20 retrospective human-review table
# ---------------------------------------------------------

top20 = ranked_queue.head(20).copy()

def confidence_note_for_row(row):
    active_reasons = int(
        row["flag_high_demand"]
        + row["flag_position_opportunity"]
        + row["flag_low_ctr"]
        + row["flag_position_instability"]
    )

    return (
        f"{row['confidence']} confidence; "
        f"{active_reasons} rule conditions fired"
    )


def what_could_make_it_wrong(row):
    risks = []

    if row["flag_low_ctr"] == 1:
        risks.append(
            "Low CTR may be normal for its position, query intent, "
            "or SERP layout."
        )

    if row["flag_position_opportunity"] == 1:
        risks.append(
            "Position 4–20 does not by itself prove that the page "
            "has a correctable content gap."
        )

    if row["flag_position_instability"] == 1:
        risks.append(
            "Position variation may reflect seasonality, changing "
            "demand, or measurement noise."
        )

    if row["flag_high_demand"] == 1:
        risks.append(
            "High demand shows materiality, not that a refresh will "
            "produce recovery."
        )

    risks.append(
        "The outcome window is short, so a temporary movement can "
        "be labelled as decline."
    )

    return " ".join(risks)


top20["confidence_note"] = top20.apply(
    confidence_note_for_row,
    axis=1,
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_could_make_it_wrong,
    axis=1,
)

top20["review_result"] = np.where(
    top20["is_declining_proxy"] == 1,
    "supported_by_proxy",
    "weak_pick_under_proxy",
)

top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "suggested_action",
        "reason_codes",
        "confidence_note",
        "is_declining_proxy",
        "review_result",
        "what_would_make_it_wrong",
    ]
].rename(
    columns={
        "is_declining_proxy": "observed_decline_proxy"
    }
)

display(top20_review)

correct_top20 = int(
    top20["is_declining_proxy"].sum()
)

print(
    f"Proxy-positive selections in Top-20: "
    f"{correct_top20} out of {len(top20)}"
)

print(
    "Precision@20:",
    round(precision_at_20, 4),
)

print(
    "Overall positive base rate:",
    round(base_rate, 4),
)

print(
    "Lift over base rate:",
    round(lift_at_20, 4),
)

,rank,client_hash_id,content_hash_id,baseline_score,suggested_action,reason_codes,confidence_note,observed_decline_proxy,review_result,what_would_make_it_wrong
0,1,client_20259bd6705d81d4,content_7fa5790abace4589,100.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high confidence; 4 rule conditions fired,1,supported_by_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
1,2,client_23a62021009f63c4,content_e7f8bc973c5204f7,100.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high confidence; 4 rule conditions fired,0,weak_pick_under_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
2,3,client_23a62021009f63c4,content_bec346091ea3e9e7,100.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high confidence; 4 rule conditions fired,0,weak_pick_under_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
3,4,client_23a62021009f63c4,content_496ecabe1eae48bb,100.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high confidence; 4 rule conditions fired,0,weak_pick_under_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
4,5,client_23a62021009f63c4,content_7c142ec4c5fddafb,100.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high confidence; 4 rule conditions fired,0,weak_pick_under_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
5,6,client_23a62021009f63c4,content_4a48f1c5632e340b,100.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr | position_instability,high confidence; 4 rule conditions fired,1,supported_by_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
6,7,client_23a62021009f63c4,content_e8a52cf3d5988c07,90.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr,high confidence; 3 rule conditions fired,1,supported_by_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
7,8,client_23a62021009f63c4,content_5e1c049f62e33b11,90.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr,high confidence; 3 rule conditions fired,1,supported_by_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
8,9,client_e547b89c05043229,content_8e5fefdae6cea24b,90.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr,high confidence; 3 rule conditions fired,0,weak_pick_under_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."
9,10,client_e547b89c05043229,content_77276ad7a26f4905,90.0,REVIEW_SNIPPET_AND_INTENT,high_demand | position_opportunity | low_ctr,high confidence; 3 rule conditions fired,0,weak_pick_under_proxy,"Low CTR may be normal for its position, query intent, or SERP layout. Position 4–20 does not by itself prove that th..."


Proxy-positive selections in Top-20: 7 out of 20
Precision@20: 0.35
Overall positive base rate: 0.2355
Lift over base rate: 1.4861


## 4. Weak picks + leakage check

A weak pick is a top-20 selection that did not decline under the defined future proxy. Such a page may still have high demand, low CTR, an actionable position, or position instability, but those signals did not correspond to the measured outcome.

If all top-20 selections are proxy-positive, I still inspect the lowest-scoring or least-supported members of the top 20 as borderline recommendations.

The baseline scoring inputs contain only historical fields available by the end of 2026-03-21. The future decline label is used only after ranking for evaluation. No future-window measurement, copied label, product score, product decision flag, client identifier, or content identifier is used as a scoring feature.

In [ ]:
# ---------------------------------------------------------
# 1. Identify weak or borderline picks
# ---------------------------------------------------------

weak_picks = top20[
    top20["is_declining_proxy"] == 0
].copy()

if not weak_picks.empty:
    weak_picks["weak_pick_type"] = (
        "observed_false_positive"
    )

    weak_picks["why_it_looks_weak"] = (
        "The rule ranked this page in the Top-20, but the "
        "future decline proxy was 0. Historical opportunity "
        "signals did not correspond to the measured outcome."
    )

else:
    # The skill requires the top of the list to be challenged
    # even when no proxy-negative rows appear.
    weak_picks = (
        top20
        .sort_values(
            by=[
                "baseline_rule_points",
                "hist_imp_per_day",
            ],
            ascending=[True, True],
        )
        .head(min(3, len(top20)))
        .copy()
    )

    weak_picks["weak_pick_type"] = (
        "borderline_proxy_positive"
    )

    weak_picks["why_it_looks_weak"] = (
        "No proxy-negative row appeared in the Top-20, so "
        "this lower-support selection is reviewed as a "
        "borderline pick. Its apparent decline may still be "
        "temporary noise or seasonality."
    )

weak_review = weak_picks[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "reason_codes",
        "suggested_action",
        "is_declining_proxy",
        "weak_pick_type",
        "why_it_looks_weak",
    ]
]

display(weak_review)

print(
    "Observed proxy-negative Top-20 picks:",
    int((top20["is_declining_proxy"] == 0).sum()),
)

# ---------------------------------------------------------
# 2. Formal leakage audit
# ---------------------------------------------------------

BASELINE_INPUT_COLUMNS = [
    "hist_imp_per_day",
    "hist_ctr",
    "hist_avg_position",
    "hist_position_sd",
]

FORBIDDEN_COLUMNS = {
    "label_outcome_imp_per_day",
    "label_outcome_impressions",
    "is_declining_proxy",
    "leak_outcome_ratio",
    "future_impressions",
    "health_score",
    "priority_score",
    "action_type",
}

future_name_tokens = [
    "future",
    "outcome",
    "label",
    "april",
    "june",
    "leak",
]

product_name_tokens = [
    "health_score",
    "priority_score",
    "action_type",
    "quick_win",
    "needs_refresh",
]

feature_window_end = pd.Timestamp("2026-03-21")
outcome_window_start = pd.Timestamp("2026-03-22")

no_forbidden_exact_columns = (
    FORBIDDEN_COLUMNS.isdisjoint(
        set(BASELINE_INPUT_COLUMNS)
    )
)

no_future_named_inputs = not any(
    token in column.lower()
    for column in BASELINE_INPUT_COLUMNS
    for token in future_name_tokens
)

no_product_flags = not any(
    token in column.lower()
    for column in BASELINE_INPUT_COLUMNS
    for token in product_name_tokens
)

ids_not_features = (
    "client_hash_id" not in BASELINE_INPUT_COLUMNS
    and "content_hash_id" not in BASELINE_INPUT_COLUMNS
)

target_not_a_feature = (
    "is_declining_proxy"
    not in BASELINE_INPUT_COLUMNS
)

windows_do_not_overlap = (
    feature_window_end < outcome_window_start
)

export_has_no_future_label = (
    "is_declining_proxy"
    not in queue_export.columns
    and "label_outcome_imp_per_day"
    not in queue_export.columns
)

leakage_audit = pd.DataFrame(
    [
        {
            "check": "No forbidden future/label column in score",
            "passed": no_forbidden_exact_columns,
        },
        {
            "check": "No future-named scoring input",
            "passed": no_future_named_inputs,
        },
        {
            "check": "No product decision flag in score",
            "passed": no_product_flags,
        },
        {
            "check": "Pseudonymous IDs are context only",
            "passed": ids_not_features,
        },
        {
            "check": "Target is evaluation only",
            "passed": target_not_a_feature,
        },
        {
            "check": "Feature and outcome windows do not overlap",
            "passed": windows_do_not_overlap,
        },
        {
            "check": "Exported action queue excludes future label",
            "passed": export_has_no_future_label,
        },
    ]
)

display(leakage_audit)

assert leakage_audit["passed"].all(), (
    "Leakage audit failed. Inspect the failed check "
    "before continuing."
)

print("PASS: all leakage checks succeeded.")
print("Baseline scoring inputs:", BASELINE_INPUT_COLUMNS)
print("Future label used only for evaluation.")

,rank,client_hash_id,content_hash_id,baseline_score,reason_codes,suggested_action,is_declining_proxy,weak_pick_type,why_it_looks_weak
1,2,client_23a62021009f63c4,content_e7f8bc973c5204f7,100.0,high_demand | position_opportunity | low_ctr | position_instability,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
2,3,client_23a62021009f63c4,content_bec346091ea3e9e7,100.0,high_demand | position_opportunity | low_ctr | position_instability,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
3,4,client_23a62021009f63c4,content_496ecabe1eae48bb,100.0,high_demand | position_opportunity | low_ctr | position_instability,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
4,5,client_23a62021009f63c4,content_7c142ec4c5fddafb,100.0,high_demand | position_opportunity | low_ctr | position_instability,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
8,9,client_e547b89c05043229,content_8e5fefdae6cea24b,90.0,high_demand | position_opportunity | low_ctr,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
9,10,client_e547b89c05043229,content_77276ad7a26f4905,90.0,high_demand | position_opportunity | low_ctr,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
10,11,client_e547b89c05043229,content_21309e9a83c83653,90.0,high_demand | position_opportunity | low_ctr,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
12,13,client_20259bd6705d81d4,content_0adb360f9005b515,90.0,high_demand | position_opportunity | low_ctr,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
15,16,client_23a62021009f63c4,content_2690f62f39fb14fe,90.0,high_demand | position_opportunity | low_ctr,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."
16,17,client_e547b89c05043229,content_cc26620b2cbb837f,90.0,high_demand | position_opportunity | low_ctr,REVIEW_SNIPPET_AND_INTENT,0,observed_false_positive,"The rule ranked this page in the Top-20, but the future decline proxy was 0. Historical opportunity signals did not ..."


Observed proxy-negative Top-20 picks: 13


,check,passed
0,No forbidden future/label column in score,True
1,No future-named scoring input,True
2,No product decision flag in score,True
3,Pseudonymous IDs are context only,True
4,Target is evaluation only,True
5,Feature and outcome windows do not overlap,True
6,Exported action queue excludes future label,True


PASS: all leakage checks succeeded.
Baseline scoring inputs: ['hist_imp_per_day', 'hist_ctr', 'hist_avg_position', 'hist_position_sd']
Future label used only for evaluation.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] The baseline rule is transparent and uses fixed human-written points
- [x] Every ranked item has reason codes, an action, and confidence
- [x] Precision@20 is shown next to the overall positive base rate
- [x] All Top-20 selections were reviewed
- [x] At least one weak or borderline recommendation was examined
- [x] The leakage audit passes
- [x] The exported queue excludes future outcome information
- [x] No client names, URLs, titles, or private queries appear
- [x] Claims use observed, measured, directional, and decision-support language
- [x] Committed under `work/notebooks/w04_baseline_score.ipynb`